<a href="https://colab.research.google.com/github/Mbaka11/IFT6135_Rep_L/blob/main/Notes/assignment2/python-code/notebooks/assignment2_practical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IFT6135 A2026 — Assignment 2

This notebook contains both programming questions: Question 1 (LSTMs) and Question 2 (Transformers).

- Run it in Colab on a GPU: **Runtime → Change runtime type → T4 GPU**.
- Choose **File → Save a copy in Drive** before editing, so that your code is kept.
- Question 1 is graded from the cell that starts with `%%writefile lstm.py`, and Question 2 from the cells that start with `# GRADED CELL`. Do not edit or delete these first lines.

**Submit to Gradescope:**
- This notebook as .ipynb (**File → Download → Download .ipynb**), to the code assignment
- Your report for both questions as one PDF, to the report assignment

# IFT6135 A2026 — Assignment 2, Question 1
## Learning to Update Memory with LSTMs
Use the accompanying assignment for the 50-point rubric.

Choose **File → Save a copy in Drive** (or **Copy to Drive**) before editing. Your saved copy keeps the implementation you write in the cell below.

Complete only the three TODO blocks in `lstm.py`. PyTorch autograd is provided; built-in recurrent modules and dictionary-based prediction are forbidden. No model solution or private grading tests are included.


## 0. Setup
Run the notebook in order. All required source files are embedded. No Google Drive mount or external dataset is needed. CPU is supported; a GPU may also be used.

**Edit your implementation in the `%%writefile lstm.py` cell**, not a second copy in the Files panel. Running that cell writes its current contents to disk and replaces any previous `lstm.py`. Save your notebook copy before disconnecting; runtime files are temporary.


In [1]:
from pathlib import Path
import os, sys
if Path.cwd().name != 'a2_q1_workspace':
    workspace = Path.cwd() / 'a2_q1_workspace'
    workspace.mkdir(exist_ok=True)
    os.chdir(workspace)
print('Working directory:', Path.cwd())


Working directory: /content/a2_q1_workspace


In [2]:
import sys, torch
print(sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.14.1+cu130
CUDA available: True
Tesla T4


## 1. Provided data, training and public tests
Run the following three cells in order without modifying them. Each cell writes one supporting file: `memory_data.py`, `train.py`, or `test_public.py`. The source is shown below for inspection. Rerunning a cell replaces its corresponding file.


In [3]:
%%writefile memory_data.py
"""Provided data utilities; students should not modify this file.

Atomic update tokens encode a (key,value) pair. Only token IDs and sequence
lengths are model inputs. Metadata is for analysis, never for prediction.
"""
from dataclasses import dataclass
import random


@dataclass(frozen=True)
class Config:
    keys: int = 3
    values: int = 4
    updates: int = 6
    distractor_types: int = 4

    @property
    def vocab_size(self):
        return 1 + self.keys * self.values + self.keys + self.distractor_types

    def update_token(self, key, value):
        return 1 + key * self.values + value

    def query_token(self, key):
        return 1 + self.keys * self.values + key

    def distractor_token(self, index):
        return 1 + self.keys * self.values + self.keys + index

    def describe(self, tokens):
        names = []
        for token in tokens:
            if token == 0:
                names.append('PAD')
            elif token <= self.keys * self.values:
                k, v = divmod(token - 1, self.values)
                names.append(f'{chr(65+k)}={v}')
            elif token < 1 + self.keys * self.values + self.keys:
                names.append('QUERY ' + chr(65 + token - 1 - self.keys*self.values))
            else:
                names.append('D' + str(token - 1 - self.keys*self.values - self.keys))
        return ' -> '.join(names)


class MemoryDataset:
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.rows[index]


def generate(n, seed, config=Config(), distractors=(0, 2, 4, 8),
             repeats=(1, 2, 3), last_positions=(2, 3, 4), used=None, prefix='data'):
    """Generate balanced (query key, target) pairs, rejecting duplicate inputs.

    Six update tokens are followed by D distractors and one query token.
    last_positions are zero-indexed positions of the final queried-key update.
    Non-query updates occupy all remaining positions. Consecutive values for
    the queried key differ. A shared `used` set makes all splits disjoint.
    """
    if config.keys < 2 or config.values < 2:
        raise ValueError('Need at least two keys and values')
    if not distractors or min(distractors) < 0 or min(repeats) < 1:
        raise ValueError('Invalid distractor/repeat counts')
    if min(last_positions) < max(repeats)-1 or max(last_positions) >= config.updates-1:
        raise ValueError('Need room for repeats and a final non-query update')
    rng = random.Random(seed)
    used = set() if used is None else used
    pairs = [(i % config.keys, (i // config.keys) % config.values,
              last_positions[(i // (config.keys*config.values)) % len(last_positions)])
             for i in range(n)]
    rng.shuffle(pairs)
    rows = []
    for index, (key, label, last) in enumerate(pairs):
        for attempt in range(10000):
            d, r = rng.choice(distractors), rng.choice(repeats)
            positions = sorted(rng.sample(range(last), r-1)) + [last]
            vals = [label]
            for _ in range(r-1):
                vals.insert(0, rng.choice([v for v in range(config.values) if v != vals[0]]))
            updates = []
            relevant = dict(zip(positions, vals))
            for t in range(config.updates):
                if t in relevant:
                    k, v = key, relevant[t]
                else:
                    k = rng.choice([k for k in range(config.keys) if k != key])
                    v = rng.randrange(config.values)
                updates.append(config.update_token(k, v))
            seq = updates + [config.distractor_token(rng.randrange(config.distractor_types))
                             for _ in range(d)] + [config.query_token(key)]
            signature = tuple(seq)
            if signature not in used:
                used.add(signature)
                break
        else:
            raise RuntimeError('Sampling space exhausted; reduce dataset size')
        rows.append(dict(id=f'{prefix}-{index:05d}', tokens=seq, target=label,
                         query_key=key, distractors=d, repeats=r, last_position=last,
                         previous_value=vals[-2] if r > 1 else -1))
    return MemoryDataset(rows)


def collate(rows):
    import torch
    lengths = torch.tensor([len(row['tokens']) for row in rows], dtype=torch.long)
    tokens = torch.zeros(len(rows), int(lengths.max()), dtype=torch.long)
    for i, row in enumerate(rows):
        tokens[i, :lengths[i]] = torch.tensor(row['tokens'], dtype=torch.long)
    return tokens, lengths, torch.tensor([r['target'] for r in rows]), rows


def make_splits(seed=6135, train_n=6000, val_n=1200, test_n=360):
    cfg = Config()
    used = set()
    train = generate(train_n, seed, cfg, used=used, prefix='train')
    val = generate(val_n, seed+1, cfg, used=used, prefix='val')
    tests = {}
    # Balance final relevant-update positions identically across conditions.
    # Never fix one position, which would permit ignoring the query altogether.
    for d in (0, 4, 8, 16, 32):
        for r in (1, 2, 3):
            tests[f'd{d}_r{r}'] = generate(test_n, seed+100+d*3+r, cfg,
                distractors=(d,), repeats=(r,), last_positions=(2,3,4), used=used,
                prefix=f'test-d{d}-r{r}')
    return cfg, train, val, tests


Writing memory_data.py


In [4]:
%%writefile train.py
"""Provided runner. Validation selects a checkpoint; tests are evaluation only."""
import argparse
import csv
import importlib.util
import json
from pathlib import Path
import platform
import random
import time
import torch
from torch.utils.data import DataLoader
from memory_data import make_splits, collate


def sync(device):
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elif device.type == 'mps':
        torch.mps.synchronize()


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss, correct, count, stale, eligible = 0., 0, 0, 0, 0
    examples, correct_examples = [], []
    for tokens, lengths, targets, metadata in loader:
        targets = targets.to(device)
        logits = model(tokens.to(device), lengths.to(device))
        total_loss += torch.nn.functional.cross_entropy(logits, targets, reduction='sum').item()
        pred = logits.argmax(-1).cpu().tolist()
        for p, row in zip(pred, metadata):
            count += 1
            correct += p == row['target']
            if row['previous_value'] >= 0:
                eligible += 1
                stale += p == row['previous_value']
            if p == row['target'] and len(correct_examples) < 4:
                correct_examples.append(dict(row, prediction=p))
            if p != row['target'] and len(examples) < 5:
                examples.append(dict(row, prediction=p))
    return dict(loss=total_loss/count, accuracy=correct/count, n=count,
                stale_count=stale, stale_eligible=eligible,
                stale_rate=stale/eligible if eligible else None, errors=examples, correct_examples=correct_examples)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--model', default='lstm.py')
    parser.add_argument('--out', default='run')
    parser.add_argument('--device', choices=['cpu','cuda','mps'], default='cpu')
    parser.add_argument('--epochs', type=int, default=20)
    parser.add_argument('--seed', type=int, default=6135)
    parser.add_argument('--train-size', type=int, default=6000)
    parser.add_argument('--val-size', type=int, default=1200)
    parser.add_argument('--test-size', type=int, default=360)
    parser.add_argument('--threads', type=int, default=4)
    args = parser.parse_args()
    torch.set_num_threads(args.threads)
    random.seed(args.seed)
    torch.manual_seed(args.seed)
    device = torch.device(args.device)
    if device.type == 'cuda':
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.cuda.reset_peak_memory_stats()
    spec = importlib.util.spec_from_file_location('submission', args.model)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    cfg, train, val, tests = make_splits(args.seed, args.train_size, args.val_size, args.test_size)
    loader = lambda ds, shuffle=False: DataLoader(ds, batch_size=128, shuffle=shuffle,
        collate_fn=collate, num_workers=0,
        generator=torch.Generator().manual_seed(args.seed) if shuffle else None)
    train_loader, val_loader = loader(train, True), loader(val)
    model = module.MemoryLSTM(cfg.vocab_size, cfg.values).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    best_loss, best_epoch, history = float('inf'), None, []
    sync(device)
    start = time.perf_counter()
    for epoch in range(1, args.epochs+1):
        model.train()
        loss_sum, correct, n = 0., 0, 0
        sync(device)
        t0 = time.perf_counter()
        for tokens, lengths, targets, _ in train_loader:
            tokens, lengths, targets = tokens.to(device), lengths.to(device), targets.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(tokens, lengths)
            loss = torch.nn.functional.cross_entropy(logits, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
            optimizer.step()
            loss_sum += loss.item()*len(targets)
            correct += (logits.argmax(-1) == targets).sum().item()
            n += len(targets)
        validation = evaluate(model, val_loader, device)
        sync(device)
        row = dict(epoch=epoch, train_loss=loss_sum/n, train_accuracy=correct/n,
                   val_loss=validation['loss'], val_accuracy=validation['accuracy'],
                   seconds=time.perf_counter()-t0)
        history.append(row)
        print(json.dumps(row), flush=True)
        if validation['loss'] < best_loss:
            best_loss, best_epoch = validation['loss'], epoch
            torch.save(model.state_dict(), out/'checkpoint.pt')
    sync(device)
    train_seconds = time.perf_counter()-start
    model.load_state_dict(torch.load(out/'checkpoint.pt', map_location=device, weights_only=True))
    # Only now inspect held-out test sets, using the validation-selected checkpoint.
    test_results = {name: evaluate(model, loader(ds), device) for name, ds in tests.items()}
    report = dict(config=vars(args), torch_version=torch.__version__, platform=platform.platform(),
                  device_name=torch.cuda.get_device_name() if device.type=='cuda' else str(device),
                  parameters=sum(p.numel() for p in model.parameters()),
                  best_epoch=best_epoch, train_and_validation_seconds=train_seconds,
                  peak_cuda_bytes=torch.cuda.max_memory_allocated() if device.type=='cuda' else None,
                  history=history, tests=test_results)
    (out/'metrics.json').write_text(json.dumps(report, indent=2))
    with (out/'test_results.csv').open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['condition','accuracy','stale_rate','n'])
        writer.writeheader()
        for name, result in test_results.items():
            writer.writerow(dict(condition=name, **{k:result[k] for k in ['accuracy','stale_rate','n']}))
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(10,4))
    for split in ['train','val']:
        axes[0].plot([r['epoch'] for r in history], [r[split+'_loss'] for r in history], label=split)
        axes[1].plot([r['epoch'] for r in history], [r[split+'_accuracy'] for r in history], label=split)
    for ax, ylabel in zip(axes, ['Cross-entropy','Accuracy']):
        ax.set(xlabel='Epoch', ylabel=ylabel)
        ax.legend()
    fig.tight_layout()
    fig.savefig(out/'learning_curves.png', dpi=150)
    plt.close(fig)
    print(f'Best epoch: {best_epoch}; training + validation: {train_seconds:.1f}s', flush=True)


if __name__ == '__main__':
    main()


Writing train.py


In [5]:
%%writefile test_public.py
"""Public examples; passing these alone does not establish correctness."""
import unittest
import torch
from lstm import LSTMCell, MemoryLSTM


class PublicTests(unittest.TestCase):
    def test_zero_cell(self):
        cell = LSTMCell(3,5)
        with torch.no_grad():
            cell.gates.weight.zero_()
            cell.gates.bias.zero_()
        h,c=cell(torch.zeros(2,3),(torch.zeros(2,5),torch.ones(2,5)))
        torch.testing.assert_close(c,torch.full((2,5),0.5))
        torch.testing.assert_close(h,0.5*torch.tanh(c))

    def test_model_shape_and_padding(self):
        torch.manual_seed(7)
        model=MemoryLSTM(20,4)
        tokens=torch.tensor([[1,2,13],[3,14,0]])
        lengths=torch.tensor([3,2])
        logits=model(tokens,lengths)
        self.assertEqual(tuple(logits.shape),(2,4))
        torch.testing.assert_close(logits[1:2],model(tokens[1:2,:2],lengths[1:2]))
        self.assertTrue(torch.isfinite(logits).all())


if __name__=='__main__':
    unittest.main(verbosity=2)


Writing test_public.py


## Inspect the provided task

In [6]:
from memory_data import Config, generate
cfg = Config()
sample = generate(12, 7)[0]
print(cfg.describe(sample['tokens']))
print('Target:', sample['target'])
print('Previous value:', sample['previous_value'])


A=3 -> C=0 -> B=2 -> A=0 -> C=0 -> A=1 -> D0 -> D3 -> QUERY B
Target: 2
Previous value: -1


## 2. Implement the LSTM
Fill in all three TODOs below, then run the cell to create `lstm.py`.

In [7]:
%%writefile lstm.py
"""Student template. Gate order is i, f, g, o. Complete the three TODO blocks."""
import torch
from torch import nn


class LSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.gates = nn.Linear(input_size + hidden_size, 4 * hidden_size)

    def forward(self, x, state):
        """x: [B,E]; state: (h,c), each [B,H]. Return (h_new,c_new)."""
        h, c = state
        # TODO: implement cell
        # Combine the current input and previous hidden state
        combined = torch.cat((x, h), dim=1)

        # Compute all four gates input at once
        gates = self.gates(combined)

        # Split the gates into input in the required order
        a_i, a_f, a_g, a_o = gates.chunk(4, dim=1)

        # Apply the gate and nonlinearities
        i = torch.sigmoid(a_i)
        f = torch.sigmoid(a_f)
        g = torch.tanh(a_g)
        o = torch.sigmoid(a_o)

        # Update the cell state and compute new hidden state
        c_new = f * c + i * g
        h_new = o * torch.tanh(c_new)

        return h_new, c_new
        raise NotImplementedError("Complete this TODO")


class MemoryLSTM(nn.Module):
    def __init__(self, vocab_size, num_values, embedding_size=32, hidden_size=64):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, embedding_size, padding_idx=0)
        self.cell = LSTMCell(embedding_size, hidden_size)
        self.classifier = nn.Linear(hidden_size, num_values)

    def recurrent(self, embeddings, lengths):
        """[B,T,E], [B] valid lengths -> final valid hidden state [B,H].

        Right padding must not update either state. Reset both states to zero
        for each call. Batch elements are independent.
        """
        # TODO: implement recurrent
        # embeddings : [B, T, E] and lengths : [B]
        batch_size, sequence_len, _ = embeddings.shape

        # Reset both states with new_zeros that keeps the same device and dtype
        h = embeddings.new_zeros(batch_size, self.hidden_size)
        c = embeddings.new_zeros(batch_size, self.hidden_size)

        # Iterate
        for t in range(sequence_len):
            # Compute candidate states for current timestep
            h_new, c_new = self.cell(embeddings[:, t, :], (h, c))

            # Valide timestep only when t < len
            valid = (t < lengths).unsqueeze(1)

            # Keep previous states for right-paded positions
            h = torch.where(valid, h_new, h)
            c = torch.where(valid, c_new, c)

        return h
        raise NotImplementedError("Complete this TODO")

    def forward(self, tokens, lengths):
        """Long tokens [B,T], Long lengths [B] -> raw value logits [B,V]."""
        # TODO: implement model
        # Convert token ID to embeddings
        embeddings = self.embedding(tokens)

        # Compute final hidden state
        final_h = self.recurrent(embeddings, lengths)

        # Compute logits
        logits = self.classifier(final_h)

        return logits
        raise NotImplementedError("Complete this TODO")


Writing lstm.py


## Complete `lstm.py`, then run public tests
These tests are expected to fail until the TODOs are complete. Passing them alone does not establish correctness.

In [8]:
import subprocess
subprocess.run([sys.executable, '-m', 'unittest', 'test_public', '-v'], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'unittest', 'test_public', '-v'], returncode=0)

## Train once and evaluate
CPU is supported; a GPU is optional. Run the supplied script once with the assignment defaults (seed 6135, 20 epochs). It selects the checkpoint with the lowest validation loss (earlier epoch on a tie), then evaluates all 15 test conditions without updating parameters. Do not tune on test results or retrain separately for each condition.


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
subprocess.run([sys.executable, 'train.py', '--model', 'lstm.py',
                '--device', device, '--out', 'run'], check=True)


## Learning curves and memory analysis
Run the following cells to generate the figures, a results table, and two example predictions for your PDF report. You may use these outputs directly; no extra plotting implementation, parameter search, or additional experiments are required.
- `D`: number of distractor tokens after the six updates.
- `R`: number of assignments to the queried key, including its first assignment.


In [ ]:
import json
from pathlib import Path
from IPython.display import display, Image
import matplotlib.pyplot as plt
metrics = json.loads(Path('run/metrics.json').read_text())
display(Image(filename='run/learning_curves.png'))
print('Selected epoch:', metrics['best_epoch'])
print('Training + validation seconds:', metrics['train_and_validation_seconds'])
print('Parameters:', metrics['parameters'])
print('Device:', metrics['device_name'])
print('Platform:', metrics['platform'])
print('Python:', sys.version.split()[0], 'PyTorch:', metrics['torch_version'])
print('Selection rule: lowest validation loss; earlier epoch on a tie.')
fig, axes = plt.subplots(1, 2, figsize=(11,4))
ds = [0,4,8,16,32]
for r in [1,2,3]:
    axes[0].plot(ds, [metrics['tests'][f'd{d}_r{r}']['accuracy'] for d in ds], marker='o', label=f'R={r}')
    if r >= 2:
        axes[1].plot(ds, [metrics['tests'][f'd{d}_r{r}']['stale_rate'] for d in ds], marker='o', label=f'R={r}')
for ax, title in zip(axes, ['Accuracy','Stale-value rate']):
    ax.set(xlabel='Distractor tokens D', ylabel=title, ylim=(0,1))
    ax.legend()
fig.tight_layout()
fig.savefig('run/memory_analysis.png', dpi=150)
plt.show()

# Table for the report (all 15 conditions).
import csv
from IPython.display import Markdown
with open('run/test_results.csv', newline='') as handle:
    results_table = list(csv.DictReader(handle))
lines = ['| Condition | Accuracy | Stale-value rate | n |',
         '|---|---:|---:|---:|']
for row in results_table:
    stale = 'N/A' if not row['stale_rate'] else f"{float(row['stale_rate']):.4f}"
    lines.append(f"| {row['condition']} | {float(row['accuracy']):.4f} | {stale} | {row['n']} |")
display(Markdown('\n'.join(lines)))


In [ ]:
# Show two errors, preferring different types when available.
all_errors = [row for result in metrics['tests'].values() for row in result['errors']]
def is_stale(row):
    return row['previous_value'] >= 0 and row['prediction'] == row['previous_value']
shown = []
for want_stale in (True, False):
    candidate = next((row for row in all_errors if is_stale(row) == want_stale), None)
    if candidate is not None:
        shown.append(candidate)
for row in all_errors:
    if len(shown) >= 2:
        break
    if row not in shown:
        shown.append(row)
if not all_errors:
    shown = [row for result in metrics['tests'].values() for row in result['correct_examples']][:2]
    print('No errors: showing two correct examples instead. Discuss the limitations.')
elif len(shown) < 2:
    print('Only one error exists: analyze this example.')
for row in shown:
    kind = 'correct' if row['prediction'] == row['target'] else ('stale-value error' if is_stale(row) else 'other error')
    print('\n', row['id'], '|', kind)
    print(cfg.describe(row['tokens']))
    print('target', row['target'], 'prediction', row['prediction'], 'previous', row['previous_value'])


## Prepare your PDF report
Answer **Question 1, Part 2** in the assignment PDF, using your own figures, results table, and examples generated above. Refer to the assignment PDF for the full requirements, including the two-example analysis and the cases with fewer errors, and the marking scheme.


# Question 2: Word-level vs. sentence-level representation learning with transformers

You will build a small Transformer encoder and train three copies of it on the same sentences, each with a different objective:
- **Masked Language Modelling (MLM):** hide a certain percentage of words and predict them. The loss is calculated on individual words.
- **Natural Language Inference (NLI) classification:** embed two sentences and predict whether the second follows from the first (entailment, neutral or contradiction). The loss is calculated on sentence embeddings, via a classifier.
- **NLI contrastive:** pull a sentence's embedding toward a sentence it entails and push it away from all the others. The loss is calculated directly between sentence embeddings.

**Allowed:** PyTorch via autograd, `nn.Linear`, `nn.Embedding`, `nn.Dropout`, `nn.LayerNorm` and basic tensor operations.

**Not allowed:** `nn.MultiheadAttention`, `F.scaled_dot_product_attention`, `nn.TransformerEncoder(Layer)`, `F.cross_entropy`, `F.nll_loss`, `nn.CrossEntropyLoss`, `F.cosine_similarity`, pretrained models.

Fill in each `# TODO` inside the cells whose first line is `# GRADED CELL`. Do not edit or delete the `# GRADED CELL` line; put only imports, functions and classes in them. Do not use variables from other cells; keep the function names and arguments. Cells starting with `# DO NOT MODIFY` are provided code, where you don't have to fill them in.

## Setup and Imports

In [ ]:
# DO NOT MODIFY - Setup
import os, time
import torch

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}", torch.cuda.get_device_name(0) if device.type == "cuda" else "(no GPU: Runtime -> Change runtime type)")

# QUICK = True trains each model for 100 steps only, to check that everything runs
QUICK = False

## Data (No Modifications Required)

All three models train on the AllNLI corpus (SNLI + MultiNLI), about 1.1M sentences:
- **MLM** uses the sentences on their own;
- **NLI classification** uses 942k (premise, hypothesis, label) pairs;
- **NLI contrastive** uses 558k (anchor, entailment, contradiction) triplets.

Each sentence is lowercased, split into words, and each word is replaced by its id in a 20,000-word vocabulary. Ids 0-3 are special: `[PAD]`=0, `[MASK]`=1, `[CLS]`=2, `[UNK]`=3 (unknown word). Every sentence starts with `[CLS]`.

The data utilities below provide the tokenizer and vocabulary, `prepare_data` (downloads and tokenizes everything once, about 2 minutes) and `Q2Data` (holds the data and makes the training batches).

In [ ]:
#@title DO NOT MODIFY - Data utilities { display-mode: "form" }
import os
import random
import re
from collections import Counter

import numpy as np
import torch


#### Tokenizer and vocabulary ##################################################
PAD_ID, MASK_ID, CLS_ID, UNK_ID = 0, 1, 2, 3
SPECIALS = ["[PAD]", "[MASK]", "[CLS]", "[UNK]"]
MAX_LEN = 64  # tokens per sentence, including [CLS]
TOKEN_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)?|[^\sa-z0-9]")


def tokenize(text):
    """Lowercase word-level tokenizer: words, numbers and single punctuation marks."""
    return TOKEN_RE.findall(text.lower())


class Vocab:
    def __init__(self, itos):
        self.itos = list(itos)
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def encode(self, text):
        """[CLS] w1 w2 ... (truncated to MAX_LEN), unknown words -> [UNK]."""
        return ([CLS_ID] + [self.stoi.get(t, UNK_ID) for t in tokenize(text)])[:MAX_LEN]

    def decode(self, ids):
        return " ".join(self.itos[i] for i in ids if i != PAD_ID)


class SentenceTable:
    """A list of token-id sequences stored as one flat array + offsets."""

    def __init__(self, flat, offsets, texts=None):
        self.flat, self.offsets, self.texts = flat, offsets, texts

    @classmethod
    def build(cls, texts, vocab):
        seqs = [vocab.encode(t) for t in texts]
        offsets = np.zeros(len(seqs) + 1, dtype=np.int64)
        offsets[1:] = np.cumsum([len(s) for s in seqs])
        flat = np.fromiter((i for s in seqs for i in s), dtype=np.int32, count=int(offsets[-1]))
        return cls(flat, offsets, list(texts))

    def __len__(self):
        return len(self.offsets) - 1

    def __getitem__(self, i):
        return self.flat[self.offsets[i] : self.offsets[i + 1]]


#### Download and preprocess (runs once) #######################################
def prepare_data(data_dir="data", vocab_size=20_000, n_dev_sentences=5_000, seed=0):
    path = os.path.join(data_dir, "q2_data.npz")
    if os.path.exists(path):
        return
    from datasets import load_dataset  # preinstalled on Colab

    os.makedirs(data_dir, exist_ok=True)
    pc = load_dataset("sentence-transformers/all-nli", "pair-class", split="train")
    tri = load_dataset("sentence-transformers/all-nli", "triplet", split="train")
    dev = load_dataset("sentence-transformers/all-nli", "pair-class", split="dev")
    dev_tri = load_dataset("sentence-transformers/all-nli", "triplet", split="dev")
    sts = load_dataset("sentence-transformers/stsb", split="test")

    clean = lambda s: " ".join(s.split())

    def indexer():
        sent_id, texts = {}, []

        def sid(s):
            s = clean(s)
            if s not in sent_id:
                sent_id[s] = len(texts)
                texts.append(s)
            return sent_id[s]
        return sid, texts

    def index(pc, tri, sid):
        pairs = np.array([(sid(p), sid(h), y) for p, h, y in zip(pc["premise"], pc["hypothesis"], pc["label"])], dtype=np.int64)
        trip = np.array([(sid(a), sid(p), sid(n)) for a, p, n in zip(tri["anchor"], tri["positive"], tri["negative"])], dtype=np.int64)
        return pairs, trip

    sid, texts = indexer()
    pairs, triplets = index(pc, tri, sid)
    dsid, dev_texts = indexer()
    dev_pairs, dev_triplets = index(dev, dev_tri, dsid)

    counts = Counter(t for s in texts for t in tokenize(s))
    vocab = Vocab(SPECIALS + [w for w, _ in counts.most_common(vocab_size - len(SPECIALS))])
    train = SentenceTable.build(texts, vocab)

    dev_tab = SentenceTable.build(dev_texts, vocab)
    train_set = set(texts)  # masked-token accuracy: dev sentences never seen in training
    dev_mlm = [i for i, s in enumerate(dev_texts) if s not in train_set]
    random.Random(seed).shuffle(dev_mlm)
    dev_mlm = np.array(sorted(dev_mlm[:n_dev_sentences]), dtype=np.int64)
    sts_a = SentenceTable.build([clean(s) for s in sts["sentence1"]], vocab)
    sts_b = SentenceTable.build([clean(s) for s in sts["sentence2"]], vocab)

    np.savez_compressed(
        path,
        itos=np.array(vocab.itos), train_flat=train.flat, train_off=train.offsets, train_text=np.array(texts),
        pairs=pairs, triplets=triplets,
        dev_flat=dev_tab.flat, dev_off=dev_tab.offsets, dev_text=np.array(dev_tab.texts),
        dev_pairs=dev_pairs, dev_triplets=dev_triplets, dev_mlm=dev_mlm,
        sts_a_flat=sts_a.flat, sts_a_off=sts_a.offsets, sts_a_text=np.array(sts_a.texts),
        sts_b_flat=sts_b.flat, sts_b_off=sts_b.offsets, sts_b_text=np.array(sts_b.texts),
        sts_score=5 * np.array(sts["score"], dtype=np.float32),
    )
    print(f"saved {path}: {len(texts):,} sentences, {len(pairs):,} labelled pairs, "
          f"{len(triplets):,} triplets, vocab {len(vocab):,}")


#### Dataset and batches #######################################################
class Q2Data:
    """Everything needed for training and evaluation, loaded from data/q2_data.npz."""

    def __init__(self, data_dir="data"):
        z = np.load(os.path.join(data_dir, "q2_data.npz"))
        self.vocab = Vocab(z["itos"].tolist())
        self.train = SentenceTable(z["train_flat"], z["train_off"], z["train_text"])
        self.pairs, self.triplets = z["pairs"], z["triplets"]
        self.dev = SentenceTable(z["dev_flat"], z["dev_off"], z["dev_text"].tolist())
        self.dev_pairs, self.dev_triplets, self.dev_mlm = z["dev_pairs"], z["dev_triplets"], z["dev_mlm"]
        self.sts_a = SentenceTable(z["sts_a_flat"], z["sts_a_off"], z["sts_a_text"].tolist())
        self.sts_b = SentenceTable(z["sts_b_flat"], z["sts_b_off"], z["sts_b_text"].tolist())
        self.sts_score = z["sts_score"]

    def summary(self):
        lens = np.diff(self.train.offsets)
        return (f"{len(self.train):,} unique training sentences (mean {lens.mean():.1f} tokens incl. [CLS]), "
                f"{len(self.pairs):,} labelled pairs, {len(self.triplets):,} triplets, "
                f"{len(self.dev_mlm):,} held-out dev sentences, {len(self.sts_score):,} STS-B test pairs, vocab {len(self.vocab):,}")

    def batches(self, task, batch_size, device, seed=0, split="train", endless=True):
        """Stream of batches for `task`, reshuffled every epoch (endless) or one pass (endless=False)."""
        rng = np.random.default_rng(seed)
        tab = self.train if split == "train" else self.dev
        rows = {"mlm": np.arange(len(self.train)) if split == "train" else self.dev_mlm,
                "nli_cls": self.pairs if split == "train" else self.dev_pairs,
                "contrastive": self.triplets if split == "train" else self.dev_triplets}[task]
        while True:
            order = rng.permutation(len(rows))
            for i in range(0, len(rows) - batch_size + 1, batch_size):
                r = rows[order[i : i + batch_size]]
                if task == "mlm":
                    yield collate([tab[j] for j in r], device)
                elif task == "nli_cls":
                    yield {"premise": collate([tab[j] for j in r[:, 0]], device),
                           "hypothesis": collate([tab[j] for j in r[:, 1]], device),
                           "label": torch.from_numpy(r[:, 2]).to(device)}
                else:
                    yield {k: collate([tab[j] for j in r[:, c]], device)
                           for c, k in enumerate(["anchor", "positive", "negative"])}
            if not endless:
                return

In [ ]:
# DO NOT MODIFY - Load the data and show examples
# Download and tokenize the data (only the first time)
prepare_data(data_dir="data")
data = Q2Data(data_dir="data")
print(data.summary())

# One example of each kind of training data
# NLI pair (used by NLI classification): premise, hypothesis and label (entailment, neutral or contradiction)
p, h, y = next(row for row in data.pairs if row[2] == 0)  # the first entailment pair
print("\nNLI pair (classification):")
print("  premise:   ", data.train.texts[p])
print("  hypothesis:", data.train.texts[h])
print("  label:     ", ["entailment", "neutral", "contradiction"][y])

# Triplet (used by NLI contrastive): anchor, positive = a sentence it entails, negative = a sentence it contradicts
a, pos, neg = data.triplets[0]
print("\nTriplet (contrastive):")
print("  anchor:                 ", data.train.texts[a])
print("  positive (entailment):  ", data.train.texts[pos])
print("  negative (contradiction):", data.train.texts[neg])

# What the model actually receives: token ids
print("\ntoken ids: ", data.train[a].tolist())
print("decoded:   ", data.vocab.decode(data.train[a]))

## Part 1: Building the model [30 points total]

What the encoder computes:

- `token ids` (B, T)  ->  `embeddings` (B, T, D)  ->  4 encoder blocks  ->  hidden states `h` (B, T, D)
- MLM head:            `h`  ->  a score for every word at every position (B, T, V)
- sentence embedding:  average of `h` over the real tokens  ->  (B, D)

Where (B = sentences in the batch, T = tokens per sentence, D = 128 hidden size, V = 20,000 words)

### Imports and Special-Token Ids (No Modifications Required)

In [ ]:
# GRADED CELL -- do not edit or delete this line
# DO NOT MODIFY - Imports and special-token ids (graded only so that the autograder gets them; nothing to fill in)
import math

import torch
import torch.nn as nn

PAD_ID, MASK_ID, CLS_ID, UNK_ID = 0, 1, 2, 3
NUM_SPECIAL = 4  # ids 0-3 are special tokens; real words start at 4

### 1.1 Batching and Padding [2 points] (To Complete)

A batch is one (B, T) tensor, but sentences have different lengths. Padding fills the short sentences with `[PAD]` = 0 up to the longest length `T`, and `attention_mask` remembers which positions are real. Attention never looks at `[PAD]` (1.3) and sentence embeddings average only the real tokens (1.6).

Ex:
- `seqs` = [[2, 5, 6], [2, 7]]
- `input_ids` = [[2, 5, 6],[2, 7, 0]]
- `attention_mask` = [[True, True, True ],[True, True, False]]
                                                                      
You will complete:
- **`collate`**: pad a list of token-id sequences into `input_ids` and build `attention_mask`.

In [ ]:
# GRADED CELL -- do not edit or delete this line
def collate(seqs, device="cpu"):
    """
    Pad a list of token-id sequences of different lengths into one batch.

    Args:
        seqs: List of B integer sequences (numpy arrays or lists). Each starts with [CLS] and has no [PAD]
        device: Where to put the returned tensors

    Returns:
        Dictionary with
            input_ids: Long tensor of shape (B, T), sequence b in row b, followed by PAD_ID up to T
            attention_mask: Bool tensor of shape (B, T), True at real tokens, False at padding
    """
    # TODO: Pad the sequences into one (B, T) tensor and build the attention mask
    # Step 1 - Length T of the longest sequence

    # Step 2 - Tensor filled with PAD_ID, with sequence i copied into the start of row i, on `device`
    # list of B sequences -> (B, T) long
    for i, s in enumerate(seqs):
        pass

    # Step 3 - attention_mask: True at real tokens, False at [PAD]
    # (B, T) long -> (B, T) bool

    raise NotImplementedError  # remove this line once your code is written

### 1.2 Embeddings [4 points] (To Complete)

Each position gets a fixed vector of sines and cosines at different frequencies (i = 0, 1, ..., D/2 - 1):

$$PE[pos, 2i] = \sin\big(pos / 10000^{2i/D}\big), \qquad PE[pos, 2i+1] = \cos\big(pos / 10000^{2i/D}\big)$$

The input of the first block is $x = \mathrm{Dropout}\big(E[\text{ids}] \cdot \sqrt{D} + PE[:T]\big)$, where E is the (V, D) token-embedding table.

You will complete:
- **`sinusoidal_positions`**: the (T, D) table PE.
- **`Embeddings.forward`**: token embeddings × √D + positions, then dropout. (`Embeddings.__init__` is provided: the token-embedding table `tok_emb`, the position table `pos` built with your `sinusoidal_positions`, and dropout.)

In [ ]:
# GRADED CELL -- do not edit or delete this line
def sinusoidal_positions(T, D):
    """
    Fixed sinusoidal position encodings.

    Args:
        T: Number of positions
        D: Model dimension (even)

    Returns:
        Float tensor of shape (T, D)
    """
    # TODO: Build the (T, D) table of sines and cosines
    # Step 1 - Positions pos = 0, ..., T-1 and even indices 2i = 0, 2, ..., D-2
    # pos: (T, 1), 2i: (1, D/2)

    # Step 2 - Angles pos / 10000^(2i / D)
    # (T, 1) and (1, D/2) -> (T, D/2)

    # Step 3 - sin of the angles in the even columns, cos in the odd columns
    # (T, D/2) -> (T, D)

    raise NotImplementedError  # remove this line once your code is written

class Embeddings(nn.Module):
    """
    Token embeddings + fixed sinusoidal positions.

    Args:
        vocab_size: Vocabulary size V
        D: Model dimension
        max_len: Number of rows of the position table
        dropout: Dropout probability
    """

    def __init__(self, vocab_size, D, max_len=128, dropout=0.1):
        # DO NOT MODIFY - Embedding layers and positional buffer
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, D)
        nn.init.normal_(self.tok_emb.weight, std=D ** -0.5)
        self.register_buffer("pos", sinusoidal_positions(max_len, D), persistent=False)
        self.scale = math.sqrt(D)
        self.drop = nn.Dropout(dropout)

    def forward(self, ids):
        """
        Forward pass.

        Args:
            ids: Token ids of shape (B, T)

        Returns:
            Input of the first encoder block, shape (B, T, D)
        """
        # TODO: Token embeddings times sqrt(D), plus positions, then dropout
        # Step 1 - Look up the token embeddings and multiply them by self.scale
        # (B, T) -> (B, T, D)

        # Step 2 - Add the first T rows of the position table self.pos
        # (B, T, D) + (T, D) -> (B, T, D)

        # Step 3 - Apply dropout
        # (B, T, D) -> (B, T, D)

        raise NotImplementedError  # remove this line once your code is written

### 1.3 Multi-Head Attention and the Bidirectional Mask [7 points] (To Complete)

For each head h = 1, ..., H, with d = D / H:

$$Q_h = W_Q^{(h)}(x),\quad K_h = W_K^{(h)}(x),\quad V_h = W_V^{(h)}(x), \qquad A_h = \mathrm{softmax}\Big(\frac{Q_h K_h^\top}{\sqrt{d}} + M\Big)\, V_h, \qquad \mathrm{MHA}(x) = W_O\big([A_1, \dots, A_H]\big)$$

The mask is 0 where query t may look at key s, and -10^4 where it may not (its softmax weight becomes ~0). GPT uses a causal mask (only s ≤ t). Ours is bidirectional: every token may look at every real token, before or after it, but never at `[PAD]`.

You will complete:
- **`MultiHeadAttention.forward`**: the formula above, one head at a time. (`MultiHeadAttention.__init__` is provided: one `nn.Linear(D, d)` per head for Q, K and V (`W_q[h]`, `W_k[h]`, `W_v[h]`), the output projection `W_o` and dropout.)
- **`bidirectional_mask`**: the (B, T, T) mask, from the (B, T) `attention_mask`.

In [ ]:
# GRADED CELL -- do not edit or delete this line
class MultiHeadAttention(nn.Module):
    """
    Multi-head self-attention with one small set of projections per head.

    Args:
        D: Model dimension
        H: Number of heads
        dropout: Dropout probability
    """

    def __init__(self, D, H, dropout=0.1):
        # DO NOT MODIFY - Per-head projections
        super().__init__()
        assert D % H == 0
        self.H, self.d = H, D // H
        self.W_q = nn.ModuleList([nn.Linear(D, self.d) for _ in range(H)])
        self.W_k = nn.ModuleList([nn.Linear(D, self.d) for _ in range(H)])
        self.W_v = nn.ModuleList([nn.Linear(D, self.d) for _ in range(H)])
        self.W_o = nn.Linear(D, D)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """
        Forward pass.

        Args:
            x: Input tensor of shape (B, T, D)
            mask: Bool tensor of shape (B, T, T) or None. mask[b, t, s] is True if query t may attend to key s

        Returns:
            Output tensor of shape (B, T, D)
        """
        # TODO: Compute every head, concatenate the heads and apply W_o
        heads = []
        for h in range(self.H):
            # Step 1 - Project x with this head's W_q[h], W_k[h] and W_v[h]
            # (B, T, D) -> q, k, v (B, T, d)

            # Step 2 - Scaled dot-product scores q k^T / sqrt(d)
            # (B, T, d) @ (B, d, T) -> (B, T, T)

            # Step 3 - Set the scores to -1e4 where the mask is False
            # (B, T, T) -> (B, T, T)
            if mask is not None:
                pass

            # Step 4 - Softmax over the keys, dropout, then weight the values; add the result to heads
            # (B, T, T) @ (B, T, d) -> (B, T, d)

        # Step 5 - Concatenate the H heads and apply the output projection self.W_o
        # H x (B, T, d) -> (B, T, D) -> (B, T, D)

        raise NotImplementedError  # remove this line once your code is written

def bidirectional_mask(attention_mask):
    """
    Attention mask of an encoder: every query may attend to every real (non-padding) key, before or after it.

    Args:
        attention_mask: Bool tensor of shape (B, T), True at real tokens, False at [PAD]

    Returns:
        Bool tensor of shape (B, T, T)
    """
    # TODO: Repeat the (B, T) key mask for every query position
    # Step 1 - Add a query dimension
    # (B, T) -> (B, 1, T)
    # Step 2 - Repeat it for each of the T query positions
    # (B, 1, T) -> (B, T, T)

    raise NotImplementedError  # remove this line once your code is written

### Feed-Forward Sublayer (No Modifications Required)

The feed-forward sublayer used in every encoder block: Linear(D, 512) → GELU → Linear(512, D).

In [ ]:
# GRADED CELL -- do not edit or delete this line
# DO NOT MODIFY - Feed-forward sublayer
class FeedForward(nn.Module):
    def __init__(self, D, F_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(D, F_dim), nn.GELU(), nn.Linear(F_dim, D))

    def forward(self, x):
        return self.net(x)

### 1.4 Encoder Block and Encoder [8 points] (To Complete)

Block:

$$x \leftarrow \mathrm{LN}_1\big(x + \mathrm{Dropout}(\mathrm{MHA}(x))\big), \qquad x \leftarrow \mathrm{LN}_2\big(x + \mathrm{Dropout}(\mathrm{FFN}(x))\big)$$

Encoder: embed, run the blocks in order, then the MLM head. The MLM head reuses the embedding table E (V, D) ("tied" weights): the score of word w at position t is $h_t \cdot E_w + b_w$, so `logits = h @ E.T + mlm_bias`, shape (B, T, V). A word gets a high score when the hidden state is close to that word's embedding.

You will complete:
- **`EncoderBlock.forward`**: the two post-norm sublayers.
- **`Encoder.forward`**: mask, embeddings, blocks and the tied MLM head.

(`EncoderBlock.__init__` and `Encoder.__init__` are provided: the sublayers, the blocks, `mlm_bias` and the NLI head.)

In [ ]:
# GRADED CELL -- do not edit or delete this line
class EncoderBlock(nn.Module):
    """
    Post-norm Transformer encoder block (as in the original Transformer and BERT).

    Args:
        D: Model dimension
        H: Number of attention heads
        F_dim: Hidden dimension of the feed-forward sublayer
        dropout: Dropout probability
    """

    def __init__(self, D, H, F_dim, dropout=0.1):
        # DO NOT MODIFY - Sublayers of the block
        super().__init__()
        self.attn = MultiHeadAttention(D, H, dropout)
        self.ffn = FeedForward(D, F_dim)
        self.norm1, self.norm2 = nn.LayerNorm(D), nn.LayerNorm(D)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask):
        """
        Forward pass.

        Args:
            x: Input tensor of shape (B, T, D)
            mask: Bool tensor of shape (B, T, T), from bidirectional_mask

        Returns:
            Output tensor of shape (B, T, D)
        """
        # TODO: The two post-norm sublayers
        # Step 1 - Attention sublayer: attention, dropout, add the residual x, then norm1
        # (B, T, D) -> (B, T, D)

        # Step 2 - Feed-forward sublayer: feed-forward, dropout, add the residual, then norm2
        # (B, T, D) -> (B, T, D)

        raise NotImplementedError  # remove this line once your code is written

class Encoder(nn.Module):
    """
    Transformer encoder: embeddings, then L encoder blocks, with two output heads.

    Args:
        vocab_size: Vocabulary size V
        D: Model dimension
        H: Number of attention heads
        L: Number of encoder blocks
        F_dim: Feed-forward hidden dimension
        max_len: Maximum sequence length
        dropout: Dropout probability
    """

    def __init__(self, vocab_size, D=128, H=4, L=4, F_dim=512, max_len=128, dropout=0.1):
        # DO NOT MODIFY - Embeddings, blocks and the two heads
        super().__init__()
        self.embed = Embeddings(vocab_size, D, max_len, dropout)
        self.blocks = nn.ModuleList([EncoderBlock(D, H, F_dim, dropout) for _ in range(L)])
        self.mlm_bias = nn.Parameter(torch.zeros(vocab_size))
        self.nli_head = nn.Linear(3 * D, 3)

    def forward(self, ids, attention_mask, return_logits=True):
        """
        Forward pass.

        Args:
            ids: Token ids of shape (B, T)
            attention_mask: Bool tensor of shape (B, T), True at real tokens
            return_logits: Compute the MLM logits (skipped when only sentence embeddings are needed)

        Returns:
            Dictionary with
                hidden: Output of the last block, shape (B, T, D)
                logits: MLM scores of shape (B, T, V), only if return_logits
        """
        # TODO: Attention mask, embeddings, encoder blocks and the tied MLM head
        # Step 1 - Attention mask from attention_mask
        # (B, T) -> (B, T, T)

        # Step 2 - Embed the ids, then apply every block of self.blocks in order
        # (B, T) -> (B, T, D) -> (B, T, D)
        for blk in self.blocks:
            pass

        # Step 3 - If return_logits, the tied MLM head: x @ E^T + self.mlm_bias, with E = self.embed.tok_emb.weight
        # (B, T, D) @ (D, V) -> (B, T, V)
        if return_logits:
            pass

        # Step 4 - Return {"hidden": ..., "logits": ...}

        raise NotImplementedError  # remove this line once your code is written

### 1.5 Masked Language Modeling [9 points] (To Complete)

Corruption (BERT): select each real word with probability 0.15 (never `[PAD]` or `[CLS]`). A selected word becomes `[MASK]` 80% of the time, a random word 10% of the time, and stays unchanged 10% of the time. The model must predict the original word at every selected position; labels are -100 everywhere else. (With `[MASK]` only, the model would learn nothing useful for the other positions, and `[MASK]` never appears in real text.)

Cross-entropy, written by hand (all three objectives use it). For scores z (N, C) and targets y (N,), averaged over the rows K where $y_n \ne -100$:

$$\mathcal{L} = -\frac{1}{|K|}\sum_{n \in K}\Big(z_{n, y_n} - \log\sum_{c} e^{z_{n,c}}\Big)$$

You will complete:
- **`mask_tokens`**: the 80 / 10 / 10 corruption and the labels.
- **`cross_entropy`**: the formula above (use `torch.logsumexp` for the log-sum, it is numerically stable).
- **`mlm_loss`**: reshape logits (B, T, V) → (B·T, V) and labels (B, T) → (B·T,), then call `cross_entropy`.

In [ ]:
# GRADED CELL -- do not edit or delete this line
def mask_tokens(ids, attention_mask, vocab_size, p=0.15):
    """
    BERT corruption for masked language modeling.

    Args:
        ids: Original token ids of shape (B, T)
        attention_mask: Bool tensor of shape (B, T), True at real tokens
        vocab_size: Number of ids V (random words are drawn from [NUM_SPECIAL, V))
        p: Selection probability

    Returns:
        inputs: Corrupted ids the model sees, shape (B, T)
        labels: Original id at selected positions, -100 elsewhere, shape (B, T)
    """
    # TODO: Choose the positions to predict, then corrupt them 80% [MASK] / 10% random word / 10% unchanged
    # Step 1 - Select each position with probability p, only at real tokens that are not special ids
    # (B, T) -> selected (B, T) bool

    # Step 2 - Labels: the original id at the selected positions, -100 everywhere else
    # (B, T) -> (B, T) long

    # Step 3 - Draw a second random number r per position; on a copy of ids, set the selected positions
    #          with r < 0.8 to MASK_ID and those with 0.8 <= r < 0.9 to a random id in [NUM_SPECIAL, vocab_size)
    # (B, T) -> inputs (B, T) long

    # Step 4 - Return inputs, labels

    raise NotImplementedError  # remove this line once your code is written

def cross_entropy(logits, targets, ignore_index=-100):
    """
    Mean negative log-likelihood, written by hand.

    Args:
        logits: Scores of shape (N, C)
        targets: Correct class of each row, long tensor of shape (N,)
        ignore_index: Target value of the rows to leave out

    Returns:
        Scalar tensor
    """
    # TODO: Cross-entropy from the formula, leaving out the rows whose target is ignore_index
    # Step 1 - Keep only the rows whose target is not ignore_index
    # (N, C), (N,) -> (K, C), (K,)

    # Step 2 - Log-probabilities: the logits minus the log-sum-exp over the classes
    # (K, C) -> (K, C)

    # Step 3 - Minus the mean, over the rows, of the log-probability of the correct class
    # (K, C) -> scalar

    raise NotImplementedError  # remove this line once your code is written

def mlm_loss(logits, labels):
    """
    Cross-entropy over the selected (masked) positions only.

    Args:
        logits: MLM scores of shape (B, T, V)
        labels: Long tensor of shape (B, T), -100 at positions to ignore

    Returns:
        Scalar tensor
    """
    # TODO: Cross-entropy over all positions at once (cross_entropy leaves out the -100 labels)
    # Step 1 - Flatten the batch and position dimensions
    # logits (B, T, V) -> (B*T, V), labels (B, T) -> (B*T,)

    # Step 2 - Return cross_entropy of the flattened tensors
    # (B*T, V), (B*T,) -> scalar

    raise NotImplementedError  # remove this line once your code is written

### 1.6 Sentence-Level Objectives (No Modifications Required)

Read these functions; you will use them in Part 2.

Cosine similarity: $\cos(u, v) = \dfrac{u \cdot v}{\lVert u\rVert\,\lVert v\rVert}$. It is 1 when u and v point the same way, 0 when they are perpendicular (unrelated), -1 when opposite.

- **`mean_pool`**: sentence embedding u = average of h over the real tokens. h (B, T, D) → u (B, D).
- **`sentence_embedding`**: runs the model and applies `mean_pool` to the last hidden states: the sentence embedding used by every model.
- **`pair_features`**: input of the NLI classifier, $[u,\ v,\ |u - v|]$. (B, D), (B, D) → (B, 3D).
- **`info_nce`**: the contrastive loss. For anchors $a_i$, positives $p_i$ (entailments) and hard negatives $n_i$ (contradictions), i = 1, ..., B:

$$\mathcal{L} = -\frac{1}{B}\sum_{i=1}^{B}\log\frac{e^{\cos(a_i, p_i)/\tau}}{\sum_{j=1}^{B} e^{\cos(a_i, p_j)/\tau} + \sum_{j=1}^{B} e^{\cos(a_i, n_j)/\tau}}$$

Every other sentence in the batch is a negative. In practice: build the (B, 2B) matrix of cosines between the anchors and all 2B candidates, divide by τ = 0.05, and apply cross-entropy with target i for row i ("which candidate is my entailment?").

In [ ]:
# GRADED CELL -- do not edit or delete this line
# DO NOT MODIFY - Sentence-level objectives (graded only because compute_loss uses them; nothing to fill in, but read them)
def mean_pool(h, attention_mask):
    """
    Average the hidden states of the real tokens of each sentence (padding excluded).

    Args:
        h: Hidden states of shape (B, T, D)
        attention_mask: Bool tensor of shape (B, T)

    Returns:
        Sentence embeddings of shape (B, D)
    """
    m = attention_mask[..., None].to(h.dtype)  # (B, T, 1): 1 at real tokens, 0 at padding
    return (h * m).sum(1) / m.sum(1).clamp(min=1)  # sum of real tokens / number of real tokens

def sentence_embedding(model, ids, attention_mask):
    """Mean-pooled last-layer hidden states: the sentence embedding used by every model."""
    return mean_pool(model(ids, attention_mask, return_logits=False)["hidden"], attention_mask)


def pair_features(u, v):
    """
    Sentence-BERT pair features for the NLI classifier: concat(u, v, |u - v|).

    Args:
        u, v: Sentence embeddings of shape (B, D)

    Returns:
        Pair features of shape (B, 3D)
    """
    return torch.cat([u, v, (u - v).abs()], dim=-1)

def info_nce(z_a, z_pos, z_neg=None, tau=0.05):
    """
    Contrastive (InfoNCE) loss with in-batch negatives and optional hard negatives.

    Args:
        z_a: Anchor embeddings of shape (B, D)
        z_pos: Positive (entailment) embeddings of shape (B, D)
        z_neg: Hard negative (contradiction) embeddings of shape (B, D), or None
        tau: Temperature

    Returns:
        Scalar tensor
    """
    cands = z_pos if z_neg is None else torch.cat([z_pos, z_neg], dim=0)  # (B or 2B, D)
    a = z_a / z_a.norm(dim=-1, keepdim=True)  # unit length, so a dot product is a cosine
    c = cands / cands.norm(dim=-1, keepdim=True)
    sim = a @ c.T / tau  # (B, 2B) cosine similarities, divided by the temperature
    return cross_entropy(sim, torch.arange(len(z_a), device=z_a.device))  # target for row i is column i

## Part 2: Losses and Training [6 points total]

### 2.1 Training Step [4 points] (To Complete)

`compute_loss` turns one batch into one scalar loss:

    mlm:          ids -> mask_tokens -> model -> mlm_loss
    nli_cls:      premise, hypothesis -> u, v = sentence_embedding (B, D) -> model.nli_head(pair_features(u, v)) (B, 3) -> cross_entropy with label
    contrastive:  anchor, positive, negative -> sentence_embedding (B, D) each -> info_nce

You will complete:
- **`compute_loss`**: the three cases above, using your Part 1 functions and the provided ones from 1.6.

In [ ]:
# GRADED CELL -- do not edit or delete this line
def compute_loss(model, batch, task, vocab_size, tau=0.05):
    """
    Loss of one batch for one of the three tasks.

    Args:
        model: An Encoder
        batch: Dictionary of tensors on the model's device:
            "mlm":         {"input_ids": (B, T), "attention_mask": (B, T)}
            "nli_cls":     {"premise": sentences, "hypothesis": sentences, "label": (B,)}
            "contrastive": {"anchor": sentences, "positive": sentences, "negative": sentences}
            where each `sentences` is {"input_ids": (B, T), "attention_mask": (B, T)}
        task: "mlm", "nli_cls" or "contrastive"
        vocab_size: Number of ids V, passed to mask_tokens
        tau: Temperature for info_nce

    Returns:
        Scalar loss (0-dim tensor, differentiable)
    """
    # TODO: The loss of one batch, for each of the three tasks
    if task == "mlm":
        # Step 1 - Corrupt the ids with mask_tokens
        # (B, T) -> inputs (B, T), labels (B, T)

        # Step 2 - Run the model on the corrupted ids and return the MLM loss of its logits
        # (B, T) -> logits (B, T, V) -> scalar
        pass

    elif task == "nli_cls":
        # Step 1 - Sentence embeddings u and v of the premise and the hypothesis (with sentence_embedding)
        # (B, T) -> u, v (B, D)

        # Step 2 - Scores of model.nli_head on pair_features(u, v); return their cross-entropy with the labels
        # (B, 3D) -> (B, 3) -> scalar
        pass

    elif task == "contrastive":
        # Step 1 - Sentence embeddings of the anchor, the positive and the negative
        # (B, T) -> (B, D) each

        # Step 2 - Return the InfoNCE loss, with temperature tau
        # (B, D) x 3 -> scalar
        pass

    else:
        raise ValueError(f"unknown task {task}")

    raise NotImplementedError  # remove this line once your code is written

### 2.2 Learning-Rate Schedule [2 points] (To Complete)

Transformers are usually trained with a short linear warmup, then a cosine decay. At the start Adam's statistics are noisy and a full learning rate can make training diverge; at the end a small learning rate lets the model settle. For step s = 0, 1, ..., S-1 with W warmup steps, the learning rate is lr · m(s):

$$m(s) = \frac{s+1}{W} \ \text{ if } s < W, \qquad m(s) = \tfrac12\Big(1 + \cos\big(\pi\,\tfrac{s - W}{S - W}\big)\Big) \ \text{ otherwise}$$

You will complete:
- **`lr_schedule`**: the multiplier m(s).

In [ ]:
# GRADED CELL -- do not edit or delete this line
def lr_schedule(step, total_steps, warmup_steps):
    """
    Learning-rate multiplier (between 0 and 1) for optimizer step `step` = 0, 1, ..., total_steps - 1.

    Args:
        step: Current optimizer step (starts at 0)
        total_steps: Number of training steps S
        warmup_steps: Number of warmup steps W (W >= 1)

    Returns:
        The multiplier m(step), a Python float
    """
    # TODO: Linear warmup, then cosine decay
    if step < warmup_steps:
        # Step 1 - Linear warmup: rises from 1 / W at step 0 to 1 at step W - 1
        pass

    # Step 2 - Cosine decay: from 1 down to 0 over the remaining S - W steps

    raise NotImplementedError  # remove this line once your code is written

### 2.3 Training (No Modifications Required)

Each model trains for a few thousand steps (about 5 minutes each on a T4) with AdamW, batch size 128 and your schedule. Model size: D = 128, 4 heads, 4 blocks (3.4M parameters). Do not change the settings. The training and validation losses are printed every 250 steps.

The training loop below provides the configuration `CONFIG`, `build_model`, `validation_loss` and `train(task, data, device)`, which builds a fresh model, optimizes your `compute_loss` and returns the model and its loss history.

### Training tips
Before the full run, set `QUICK = True` in the setup cell and run everything once: each model then trains for 100 steps only, which checks in about a minute that your code runs without errors and that the losses decrease. Then set `QUICK = False` and run all the cells again.

In [ ]:
#@title DO NOT MODIFY - Training loop { display-mode: "form" }
import time

import torch


#### Configuration (do not change) #############################################
CONFIG = dict(
    D=128, H=4, L=4, F_dim=512, dropout=0.1,
    batch_size=128, lr=5e-4, weight_decay=0.01, warmup_frac=0.05, grad_clip=1.0, tau=0.05,
    steps={"mlm": 6000, "nli_cls": 5000, "contrastive": 4000},
    eval_every=250, val_batches=20, seed=0,
)


#### Model #####################################################################
def build_model(vocab_size, cfg=CONFIG):
    return Encoder(vocab_size, cfg["D"], cfg["H"], cfg["L"], cfg["F_dim"], dropout=cfg["dropout"])


def use_amp(device):
    """Mixed precision only where it is fast (tensor cores: T4, V100, A100, ...)."""
    return device.type == "cuda" and torch.cuda.get_device_capability(device)[0] >= 7


#### Validation loss ###########################################################
@torch.no_grad()
def validation_loss(model, data, task, device, cfg=CONFIG):
    """Mean loss over the first cfg['val_batches'] dev batches, with fixed corruption for MLM."""
    was_training = model.training
    model.eval()
    cpu_state = torch.get_rng_state()
    cuda_state = torch.cuda.get_rng_state(device) if device.type == "cuda" else None
    torch.manual_seed(1234)
    losses = []
    for i, batch in enumerate(data.batches(task, cfg["batch_size"], device, seed=0, split="dev", endless=False)):
        if i == cfg["val_batches"]:
            break
        losses.append(compute_loss(model, batch, task, len(data.vocab), cfg["tau"]).item())
    torch.set_rng_state(cpu_state)
    if cuda_state is not None:
        torch.cuda.set_rng_state(cuda_state, device)
    model.train(was_training)
    return sum(losses) / len(losses)


#### Training loop #############################################################
def train(task, data, device, cfg=CONFIG, steps=None, log=print):
    """Train one model on `task` in {"mlm", "nli_cls", "contrastive"}.

    Returns: model, history = {"step", "train_loss", "val_step", "val_loss", "time"}.
    """
    torch.manual_seed(cfg["seed"])
    model = build_model(len(data.vocab), cfg).to(device)
    steps = steps or cfg["steps"][task]
    amp = use_amp(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    warm = max(1, int(cfg["warmup_frac"] * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: lr_schedule(s, steps, warm))  # Part 2.2
    scaler = torch.amp.GradScaler("cuda", enabled=amp)

    hist = {"task": task, "step": [], "train_loss": [], "val_step": [0], "val_loss": [], "time": 0.0}
    hist["val_loss"].append(validation_loss(model, data, task, device, cfg))
    log(f"[{task}] step 0  val loss {hist['val_loss'][-1]:.3f}  (amp={amp})")
    t0, running = time.time(), []
    model.train()
    for step, batch in enumerate(data.batches(task, cfg["batch_size"], device, seed=cfg["seed"]), start=1):
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            loss = compute_loss(model, batch, task, len(data.vocab), cfg["tau"])
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
        scaler.step(opt)
        scaler.update()
        sched.step()
        running.append(loss.item())
        if step % 50 == 0 or step == steps:
            hist["step"].append(step)
            hist["train_loss"].append(sum(running) / len(running))
            running = []
        if step % cfg["eval_every"] == 0 or step == steps:
            hist["val_step"].append(step)
            hist["val_loss"].append(validation_loss(model, data, task, device, cfg))
            log(f"[{task}] step {step}/{steps}  train loss {hist['train_loss'][-1]:.3f}  "
                f"val loss {hist['val_loss'][-1]:.3f}  ({time.time() - t0:.0f}s)")
        if step == steps:
            break
    hist["time"] = time.time() - t0
    model.eval()
    return model, hist

In [ ]:
# DO NOT MODIFY - Configuration and untrained baseline model
print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

os.makedirs("outputs", exist_ok=True)
models, histories = {}, {}

# Untrained baseline: same architecture, random weights, never trained
torch.manual_seed(0)
models["random"] = build_model(len(data.vocab)).to(device).eval()
num_params = sum(p.numel() for p in models["random"].parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {num_params:,}")

In [ ]:
# DO NOT MODIFY - Train the MLM model
print("Training the MLM model...")
models["mlm"], histories["mlm"] = train(task="mlm", data=data, device=device, steps=100 if QUICK else None)
torch.save(models["mlm"].state_dict(), "outputs/mlm.pt")

In [ ]:
# DO NOT MODIFY - Train the NLI classification model
print("Training the NLI classification model...")
models["nli_cls"], histories["nli_cls"] = train(task="nli_cls", data=data, device=device, steps=100 if QUICK else None)
torch.save(models["nli_cls"].state_dict(), "outputs/nli_cls.pt")

In [ ]:
# DO NOT MODIFY - Train the NLI contrastive model
print("Training the NLI contrastive model...")
models["contrastive"], histories["contrastive"] = train(task="contrastive", data=data, device=device, steps=100 if QUICK else None)
torch.save(models["contrastive"].state_dict(), "outputs/contrastive.pt")

## Part 3: Evaluation [14 points total]

Answer the Part 3 questions in your PDF report, using the outputs of the cells below.

### Evaluation (No Modifications Required)

The three trained models and the untrained one are compared in two ways.

**Token level: masked-word accuracy.** Hide 15% of the words in 5,000 sentences no model trained on, and count how often the MLM head's top guess is the original word.

**Sentence level: cosine of similar vs. dissimilar pairs.** The cosine similarity of two sentence embeddings u and v is

$$\cos(u, v) = \frac{u \cdot v}{\lVert u\rVert\,\lVert v\rVert}$$

It is 1 when u and v point the same way, 0 when they are perpendicular (unrelated), and -1 when they point in opposite directions. STS-B is a set of 1,379 sentence pairs that people scored from 0 (different meaning) to 5 (same meaning). We take the 338 pairs scored 4 or more ("similar") and the 308 pairs scored 1 or less ("dissimilar"), and average the cosine between the two sentence embeddings of each pair. A good sentence embedding gives similar pairs a much higher cosine than dissimilar ones: a large **gap**.

The evaluation utilities below provide the two metrics above, the NLI classifier's accuracy, the results table, the plots and the two "Try it" demos.

In [ ]:
#@title DO NOT MODIFY - Evaluation utilities { display-mode: "form" }
import matplotlib.pyplot as plt
import numpy as np
import torch


#### Names and colors ##########################################################
LABELS = {"random": "Random init", "mlm": "MLM", "nli_cls": "NLI classification", "contrastive": "NLI contrastive"}
COLORS = {"random": "#9e9e9e", "mlm": "#1f77b4", "nli_cls": "#ff7f0e", "contrastive": "#2ca02c"}


#### Token level: masked-word accuracy #########################################
@torch.no_grad()
def masked_token_accuracy(model, data, device, batch_size=256):
    model.eval()
    torch.manual_seed(1234)
    correct = total = 0
    for b in data.batches("mlm", batch_size, device, split="dev", endless=False):
        inputs, labels = mask_tokens(b["input_ids"], b["attention_mask"], len(data.vocab))
        keep = labels != -100
        h = model(inputs, b["attention_mask"], return_logits=False)["hidden"][keep]
        logits = h @ model.embed.tok_emb.weight.T + model.mlm_bias  # MLM head at the masked positions only
        correct += (logits.argmax(-1) == labels[keep]).sum().item()
        total += keep.sum().item()
    return correct / total


#### Sentence level: cosine of similar vs. dissimilar STS-B pairs ##############
@torch.no_grad()
def embed_table(model, table, device, batch_size=512):
    model.eval()
    out = []
    for i in range(0, len(table), batch_size):
        b = collate([table[j] for j in range(i, min(i + batch_size, len(table)))], device)
        out.append(sentence_embedding(model, b["input_ids"], b["attention_mask"]).float().cpu())
    return torch.cat(out)


def cosine(a, b):
    a = a / a.norm(dim=-1, keepdim=True)
    b = b / b.norm(dim=-1, keepdim=True)
    return (a * b).sum(-1).numpy()


SIMILAR, DISSIMILAR = 4.0, 1.0  # STS-B human scores (0-5) that count as similar / dissimilar


def similar_dissimilar(model, data, device):
    """Cosines of the STS-B test pairs scored >= 4 (similar) and <= 1 (dissimilar) by humans."""
    ea, eb = embed_table(model, data.sts_a, device), embed_table(model, data.sts_b, device)
    cos, score = cosine(ea, eb), np.asarray(data.sts_score)
    return cos[score >= SIMILAR], cos[score <= DISSIMILAR]


#### NLI classifier accuracy ###################################################
@torch.no_grad()
def nli_accuracy(model, data, device, batch_size=256):
    model.eval()
    correct = total = 0
    for b in data.batches("nli_cls", batch_size, device, split="dev", endless=False):
        u = sentence_embedding(model, b["premise"]["input_ids"], b["premise"]["attention_mask"])
        v = sentence_embedding(model, b["hypothesis"]["input_ids"], b["hypothesis"]["attention_mask"])
        pred = model.nli_head(pair_features(u, v)).argmax(-1)
        correct += (pred == b["label"]).sum().item()
        total += len(pred)
    return correct / total


#### All metrics, table and plots ##############################################
def evaluate_all(models, data, device):
    """models: {"random": model, "mlm": model, ...}. Returns a dict of results per model."""
    res = {}
    for name, model in models.items():
        sim, dis = similar_dissimilar(model, data, device)
        res[name] = {
            "masked_token_acc": masked_token_accuracy(model, data, device),
            "cos_similar": float(sim.mean()),
            "cos_dissimilar": float(dis.mean()),
            "gap": float(sim.mean() - dis.mean()),
            "nli_dev_acc": nli_accuracy(model, data, device) if name == "nli_cls" else None,
        }
    return res


def results_text(res):
    """Plain-text results table for the notebook."""
    lines = [f"{'':20s}{'masked-word acc.':>18s}{'cos similar':>14s}{'cos dissimilar':>16s}{'gap':>7s}"]
    for name in LABELS:
        if name in res:
            r = res[name]
            lines.append(f"{LABELS[name]:20s}{100 * r['masked_token_acc']:17.1f}%{r['cos_similar']:14.2f}"
                         f"{r['cos_dissimilar']:16.2f}{r['gap']:7.2f}")
    return "\n".join(lines)


def plot_loss_curves(histories, path):
    fig, axes = plt.subplots(1, len(histories), figsize=(4.2 * len(histories), 3.2))
    for ax, (name, h) in zip(np.atleast_1d(axes), histories.items()):
        ax.plot(h["step"], h["train_loss"], color=COLORS[name], alpha=0.5, label="train")
        ax.plot(h["val_step"], h["val_loss"], color=COLORS[name], marker="o", ms=3, label="validation")
        ax.set_title(LABELS[name])
        ax.set_xlabel("step")
        ax.set_ylabel("loss")
        ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)


def plot_results(res, path):
    """Left: masked-word accuracy (token level). Right: mean cosine of similar vs dissimilar STS-B pairs."""
    names = [n for n in LABELS if n in res]
    x = np.arange(len(names))
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.4))
    a1.bar(x, [100 * res[n]["masked_token_acc"] for n in names], color=[COLORS[n] for n in names])
    a1.set_title("Token level: masked-word accuracy (%)")
    a2.bar(x - 0.2, [res[n]["cos_similar"] for n in names], 0.4, color=[COLORS[n] for n in names], label="similar pairs")
    a2.bar(x + 0.2, [res[n]["cos_dissimilar"] for n in names], 0.4, color=[COLORS[n] for n in names], alpha=0.4,
           hatch="//", label="dissimilar pairs")
    for i, n in enumerate(names):
        a2.text(i, res[n]["cos_similar"] + 0.03, f"gap {res[n]['gap']:.2f}", ha="center", fontsize=8)
    a2.set_ylim(0, 1.25)
    a2.set_title("Sentence level: mean cosine of STS-B pairs")
    a2.legend(fontsize=8, loc="upper center", ncol=2)
    for a in (a1, a2):
        a.set_xticks(x)
        a.set_xticklabels([LABELS[n].replace(" ", "\n", 1) for n in names], fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)


#### Demos: fill in the blank, search by meaning ###############################
@torch.no_grad()
def fill_in_blank(models, data, device, text, k=5):
    """Top-k words each model predicts for the single [MASK] in `text`."""
    words = tokenize(text.replace("[MASK]", " zzmaskzz "))
    ids = [CLS_ID] + [MASK_ID if w == "zzmaskzz" else data.vocab.stoi.get(w, UNK_ID) for w in words]
    pos = ids.index(MASK_ID)
    b = collate([ids], device)
    out = [text]
    for name, model in models.items():
        model.eval()
        logits = model(b["input_ids"], b["attention_mask"])["logits"][0, pos]
        logits[: NUM_SPECIAL] = -float("inf")  # only real words
        top = logits.topk(k).indices.tolist()
        out.append(f"  {LABELS[name]:20s} " + ", ".join(data.vocab.itos[i] for i in top))
    return "\n".join(out)


def search(models, data, device, query, k=3):
    """The k STS-B test sentences closest (cosine) to `query`, for each model."""
    texts = list(dict.fromkeys(data.sts_a.texts + data.sts_b.texts))  # unique sentences
    table = [data.vocab.encode(t) for t in texts]
    q = [data.vocab.encode(query)]
    out = [f"query: {query}  (searching {len(texts)} sentences)"]
    for name, model in models.items():
        e = embed_table(model, table, device)
        e = e / e.norm(dim=-1, keepdim=True)
        eq = embed_table(model, q, device)[0]
        sims = e @ (eq / eq.norm())
        out.append(f"== {LABELS[name]}")
        out += [f"   {sims[j]:.2f}  {texts[j]}" for j in sims.argsort(descending=True)[:k].tolist()]
    return "\n".join(out)

In [ ]:
# DO NOT MODIFY - Loss curves (training and validation loss of each model)
from IPython.display import Image, display
plot_loss_curves(histories, path="outputs/loss_curves.png")
display(Image("outputs/loss_curves.png"))

In [ ]:
# DO NOT MODIFY - Evaluate all models: results table, masked-word accuracy plot and cosine plot
res = evaluate_all(models, data, device)
print(results_text(res))
print("\nNLI classifier accuracy on held-out pairs: %.1f%% (3 classes, chance = 33%%)" % (100 * res["nli_cls"]["nli_dev_acc"]))
plot_results(res, path="outputs/results.png")
display(Image("outputs/results.png"))

### Try It: Fill in the Blank

Each model's MLM head guesses the hidden word (top 5 guesses). Keep the two given sentences and add one of your own (the report asks for all three). Each sentence needs exactly one `[MASK]` and lowercase words.

In [ ]:
# YOU MAY EDIT - add one sentence of your own after the two given ones
for text in ["a man is playing a [MASK] .", "two dogs are [MASK] in the snow ."]:
    print(fill_in_blank(models, data, device, text), "\n")

### Try It: Search by Meaning

Each model embeds your query and the 2,551 STS-B sentences, and prints the 3 sentences with the highest cosine. Keep the two given queries and add one of your own (the report asks for all three).

In [ ]:
# YOU MAY EDIT - add one query of your own after the two given ones
trained = {k: models[k] for k in ["mlm", "nli_cls", "contrastive"]}
for query in ["a cat is sleeping on the couch .", "the stock market fell sharply today ."]:
    print(search(trained, data, device, query), "\n")